# 02 — Household Preprocessing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohamed-Al-Saudi/EnergySavvy-AI-Project/blob/main/notebooks/02_household_preprocessing.ipynb)

Create a reproducible cleaned dataset. Decisions should be based on EDA findings.

Preprocess UCI household dataset based on EDA findings from 01.
From EDA: 2,075,259 rows, 1.25% missing (25,979 rows), 0 duplicates, right-skewed power, Voltage ~240V, zero-inflated sub-meterings, Global_active vs Global_intensity = 1.0, peaks 8h and 20h.
This notebook is independent from Cairo weather - do NOT merge.

### 1. Load raw — Handles.zip with.txt inside

In [30]:
import pandas as pd

zip_url = "https://raw.githubusercontent.com/Mohamed-Al-Saudi/EnergySavvy-AI-Project/main/data/household_power/raw/individual+household+electric+power+consumption.zip"

print("Downloading data from UCI...")

df = pd.read_csv(
    zip_url,
    compression='zip', # <-- tells pandas it's a zip
    sep=';',
    na_values=['', '?'],
    low_memory=False
)

print(f"Shape: {df.shape}") # Should be (2075259, 9)
df.head()

Shape: (2075259, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### 2. Inspect - Confirm EDA findings

In [31]:
print(df.isna().sum()) # 25979 each
print(f"Rows any missing: {df.isna().any().sum()} - 1.25%")
print(f"Duplicates: {df.duplicated(['Date','Time']).sum()}")
df.info()

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64
Rows any missing: 7 - 1.25%
Duplicates: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), object(2)
memory usage: 142.5+ MB


### 3. Parse DateTime
Combine Date+Time -> datetime index, sort for time interpolation.

In [32]:
df['datetime'] = pd.to_datetime(df['Date']+' '+df['Time'], format='%d/%m/%Y %H:%M:%S')
df = df.set_index('datetime').sort_index().drop(columns=['Date','Time'])
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### 4. Handle Missing - 1.25% sensor outage
Use time interpolation, not mean. Mean would break time continuity.

In [33]:
print("Before:", df.isna().sum().sum())
df = df.interpolate(method='time', limit_direction='both').ffill().bfill()
print("After:", df.isna().sum().sum())

Before: 181853
After: 0


### 5. Optimize dtypes

In [34]:
for c in df.columns:
    df[c] = df[c].astype('float32')
print(f"Memory: {df.memory_usage().sum()/1024**2:.1f} MB")
df.describe().loc[['min','max']].T

Memory: 71.2 MB


,min,max
Global_active_power,0.076000,11.122000
Global_reactive_power,0.000000,1.390000
Voltage,223.199997,254.149994
Global_intensity,0.200000,48.400002
Sub_metering_1,0.000000,88.000000
Sub_metering_2,0.000000,80.000000
Sub_metering_3,0.000000,31.000000


### 6. Feature Engineering
unmeasured = Global_active*1000/60 - sum(subs)

unmeasured can be:
  - lights
  - TV
  - computers, routers
  - phone chargers
  - microwave not in kitchen circuit etc.

In [35]:
df['global_active_Wh'] = df['Global_active_power']*1000/60
df['unmeasured_Wh'] = (df['global_active_Wh'] - (df['Sub_metering_1']+df['Sub_metering_2']+df['Sub_metering_3'])).clip(lower=0)
df['hour'] = df.index.hour.astype('int8')
df['dayofweek'] = df.index.dayofweek.astype('int8')
df['month'] = df.index.month.astype('int8')
df['year'] = df.index.year.astype('int16')
df['is_weekend'] = (df['dayofweek'] >= 5).astype('int8')
df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,global_active_Wh,unmeasured_Wh,hour,dayofweek,month,year,is_weekend
datetime,,,,,,,,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.839996,18.4,0.0,1.0,17.0,70.266670,52.266670,17,5,12,2006,1
2006-12-16 17:25:00,5.360,0.436,233.630005,23.0,0.0,1.0,16.0,89.333336,72.333336,17,5,12,2006,1
2006-12-16 17:26:00,5.374,0.498,233.289993,23.0,0.0,2.0,17.0,89.566666,70.566666,17,5,12,2006,1
2006-12-16 17:27:00,5.388,0.502,233.740005,23.0,0.0,1.0,17.0,89.800003,71.800003,17,5,12,2006,1
2006-12-16 17:28:00,3.666,0.528,235.679993,15.8,0.0,1.0,17.0,61.099998,43.099998,17,5,12,2006,1


### 7. Resample - 1min noisy, need hourly for 03_forecasting
Group all rows that belong to same hour, then apply mean (2M -> 34k rows)

In [36]:
hourly = df.resample('H').agg({
    'Global_active_power':'mean','Voltage':'mean','Global_intensity':'mean',
    'Sub_metering_1':'sum','Sub_metering_2':'sum','Sub_metering_3':'sum',
    'unmeasured_Wh':'sum'
})
daily = df.resample('D').agg({'Global_active_power':'mean','Sub_metering_1':'sum','Sub_metering_2':'sum','Sub_metering_3':'sum'}).dropna()
print(f"hourly {hourly.shape}, daily {daily.shape}")
hourly.head()

/tmp/ipykernel_1008/4088396945.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = df.resample('H').agg({


hourly (34589, 7), daily (1442, 4)


,Global_active_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,unmeasured_Wh
datetime,,,,,,,
2006-12-16 17:00:00,4.222889,234.643875,18.099998,0.0,19.0,607.0,1907.733398
2006-12-16 18:00:00,3.632200,234.580154,15.600000,0.0,403.0,1012.0,2217.199951
2006-12-16 19:00:00,3.400233,233.232498,14.503333,0.0,86.0,1001.0,2313.233398
2006-12-16 20:00:00,3.268567,234.071503,13.916667,0.0,0.0,1007.0,2261.566650
2006-12-16 21:00:00,3.056467,237.158661,13.046666,0.0,25.0,1033.0,1998.466675


### 8. Save Processed

In [37]:
from pathlib import Path

PROCESSED_DIR = Path("data/household_power/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save 1-min full cleaned (for 04 anomaly if you need minute level)
df.to_parquet(PROCESSED_DIR / "household_power_1min.parquet")
print(f"Saved 1min: {df.shape} -> {PROCESSED_DIR / 'household_power_1min.parquet'}")

# 2. Save hourly - THIS is what 03,04,05 will use
hourly.to_parquet(PROCESSED_DIR / "household_power_hourly.parquet")
print(f"Saved hourly: {hourly.shape} -> {PROCESSED_DIR / 'household_power_hourly.parquet'}")

# 3. Save hourly as CSV too (for GitHub preview / Colab badge)
hourly.to_csv(PROCESSED_DIR / "household_power_hourly.csv", index=True)
print(f"Saved hourly csv")

# 4. Save daily for quick check
daily.to_csv(PROCESSED_DIR / "household_power_daily.csv")
print(f"Saved daily: {daily.shape}")

# Check files created
for f in PROCESSED_DIR.iterdir():
    print(f"  - {f.name} ({f.stat().st_size/1024:.1f} KB)")

Saved 1min: (2075259, 14) -> data/household_power/processed/household_power_1min.parquet
Saved hourly: (34589, 7) -> data/household_power/processed/household_power_hourly.parquet
Saved hourly csv
Saved daily: (1442, 4)
  - household_power_daily.csv (57.9 KB)
  - household_power_hourly.csv (2404.2 KB)
  - household_power_hourly.parquet (985.3 KB)
  - household_power_1min.parquet (34576.2 KB)


### 9. Key findings for next phase


#### 1. Data Cleaning
- Loaded **2,075,259 rows** (1-minute frequency, Dec 2006 - Nov 2010)
- Found **25,980 missing values (1.25%)** encoded as `?` -> converted to NaN
- Created `datetime` index from `Date + Time`
- Removed **0 duplicated** timestamps

#### 2. Missing Value Handling
- Strategy: **Time interpolation** (`method='time'`), NOT mean/median
- Why: Energy is time-series, gap filling must respect order
- Remaining NaNs at start/end: filled with `ffill/bfill`
- Result: 0 NaNs remaining

#### 3. Memory Optimization
- Converted all `float64 -> float32` (50% memory saved: 127 MB -> 64 MB)
- Converted `hour, dayofweek, month -> int8` and `year -> int16`

#### 4. Feature Engineering
- Split datetime index:
    - `hour (0-23)` -> Peak at **20h = 1.9 kW**, low at 4h = 0.4 kW
    - `dayofweek (0-6)` -> Weekend slightly higher than weekday
    - `month (1-12)` -> Winter (Nov-Feb) = higher consumption
    - `is_weekend` binary
- Unit fix: `Global_active_power (kW) -> Wh = kW * 1000 / 60`
- Created **unmeasured_Wh**:
    - `unmeasured = Global_active*1000/60 - (Sub1+Sub2+Sub3)`
    - Represents lights, TV, PC etc. not wired to sub-meters
    - Mean ~12 Wh/min, clipped negatives to 0

#### 5. Resampling
- **1-min:** 2,075,259 rows (noisy, for anomaly detection)
- **Hourly:** 34,589 rows (`mean` for power, `sum` for Wh) -> **Main file for 03,04,05**
- **Daily:** 1,441 rows (for long-term trend)

#### 6. Output for Next Notebooks
- Saved to `data/household_power/processed/`:
    - `household_power_1min.parquet` (full cleaned)
    - `household_power_hourly.parquet` (34k rows) -> **Input for 03_forecasting, 04_anomaly, 05_recommendation**
    - `household_power_hourly.csv` (same for Colab preview)
    - `household_power_daily.csv`

> **This file does cleaning ONCE. Files 03,04,05 will LOAD hourly.parquet, not repeat cleaning.**